# LA temperatures, summer 2025
> This notebook fetches and processes average cloudiness data captured at US weather stations from the [National Centers for Environmental Information](https://www.ncei.noaa.gov/products/land-based-station/comparative-climatic-data). [The table]('https://www.ncei.noaa.gov/pub/data/ccd-data/clpcdy20.dat') shows the historical mean number of days per category of cloudiness. The categories are determined for daylight hours only. Clear denotes zero to 3/10 average sky cover. Partly cloudy denotes 4/10 to 7/10 average sky cover. Cloudy denotes 8/10 to 10/10 average sky cover. The data are used to try to better understand the "May gray" and "June gloom" phenomena in Los Angeles.

In [1]:
import json
import requests
import pandas as pd
import jupyter_black
import altair as alt
import geopandas as gpd
import seaborn as sns

In [2]:
jupyter_black.load()
pd.options.display.max_columns = 100
pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = None

In [3]:
today = pd.Timestamp("today").strftime("%Y%m%d")

---

## Read data

In [4]:
# Define the column names based on the dataset structure
column_names = [
    "station",
    "years",
    "Jan_CL",
    "Jan_PC",
    "Jan_CD",
    "Feb_CL",
    "Feb_PC",
    "Feb_CD",
    "Mar_CL",
    "Mar_PC",
    "Mar_CD",
    "Apr_CL",
    "Apr_PC",
    "Apr_CD",
    "May_CL",
    "May_PC",
    "May_CD",
    "Jun_CL",
    "Jun_PC",
    "Jun_CD",
    "Jul_CL",
    "Jul_PC",
    "Jul_CD",
    "Aug_CL",
    "Aug_PC",
    "Aug_CD",
    "Sep_CL",
    "Sep_PC",
    "Sep_CD",
    "Oct_CL",
    "Oct_PC",
    "Oct_CD",
    "Nov_CL",
    "Nov_PC",
    "Nov_CD",
    "Dec_CL",
    "Dec_PC",
    "Dec_CD",
    "ANN_CL",
    "ANN_PC",
    "ANN_CD",
]

# Load the data into a DataFrame using fixed-width format, skipping the first two rows
url = "https://www.ncei.noaa.gov/sites/g/files/anmtlf171/files/2024-11/clpcdy23.txt"
src = pd.read_fwf(url, skiprows=2, names=column_names)

In [5]:
# Melt the DataFrame to long format for clear days
df_clear = src.melt(
    id_vars=["station", "years"],
    value_vars=[
        f"{month}_CL"
        for month in [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
            "ANN",
        ]
    ],
    var_name="month",
    value_name="days",
)

# Melt the DataFrame to long format for partly cloudy days
df_pc = src.melt(
    id_vars=["station", "years"],
    value_vars=[
        f"{month}_PC"
        for month in [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
            "ANN",
        ]
    ],
    var_name="month",
    value_name="days",
)

# Melt the DataFrame to long format for cloudy days
df_cd = src.melt(
    id_vars=["station", "years"],
    value_vars=[
        f"{month}_CD"
        for month in [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
            "ANN",
        ]
    ],
    var_name="month",
    value_name="days",
)

In [6]:
df_melted = pd.concat([df_clear, df_pc, df_cd])

In [7]:
months = {
    "Jan": "1",
    "Feb": "2",
    "Mar": "3",
    "Apr": "4",
    "May": "5",
    "Jun": "6",
    "Jul": "7",
    "Aug": "8",
    "Sep": "9",
    "Oct": "10",
    "Nov": "11",
    "Dec": "12",
    "ANN": "0",
}

In [8]:
conditions = {
    "CL": "clear",
    "PC": "partly_cloudy",
    "CD": "cloudy",
}

In [9]:
df_melted[["month_abbr", "condition_code"]] = df_melted["month"].str.split(
    "_", expand=True
)

In [10]:
df_melted["condition"] = df_melted["condition_code"].map(conditions)

In [11]:
df_melted["month"] = df_melted["month_abbr"].map(months).astype(int)

In [12]:
df_melted["station_code"] = df_melted["station"].str[:5].astype(str)

In [13]:
df_melted[["station_location", "station_state"]] = (
    df_melted["station"].str[5:].str.split(",", expand=True)
)

In [14]:
df_melted["station_state"] = df_melted["station_state"].str.strip()

In [15]:
df = (
    df_melted[
        [
            "station_code",
            "station_location",
            "station_state",
            "years",
            "month",
            "days",
            "condition",
        ]
    ]
    .query("month != 0")
    .copy()
    .reset_index(drop=True)
)

In [16]:
df.head()

,station_code,station_location,station_state,years,month,days,condition
0,13876,BIRMINGHAM AP,AL,37,1,7,clear
1,03856,HUNTSVILLE,AL,27,1,7,clear
2,13894,MOBILE,AL,47,1,8,clear
3,13895,MONTGOMERY,AL,51,1,7,clear
4,26451,ANCHORAGE,AK,44,1,7,clear


In [17]:
df_ca = df.query('station_code == "23174"').reset_index(drop=True)

In [18]:
df_ca_pivot = (
    df_ca.pivot(
        index=["station_code", "station_location", "month"],
        values="days",
        columns="condition",
    )
    .reset_index()
    .sort_values("month")
)

In [19]:
df_ca_pivot["partly_cloudy_or_cloudy"] = (
    df_ca_pivot["partly_cloudy"] + df_ca_pivot["cloudy"]
)

In [21]:
# Define the color gradient for the heatmap
cm = sns.light_palette("#f8c153", as_cmap=True, reverse=True)

# Apply the gradient to the 'partly_cloudy_or_cloudy' column
styled_df = df_ca_pivot.style.background_gradient(
    cmap=cm, subset=["partly_cloudy_or_cloudy"]
)

# Display the styled DataFrame
styled_df

condition,station_code,station_location,month,clear,cloudy,partly_cloudy,partly_cloudy_or_cloudy
0,23174,LOS ANGELES AP,1,12,11,8,19
1,23174,LOS ANGELES AP,2,11,11,6,17
2,23174,LOS ANGELES AP,3,12,11,9,20
3,23174,LOS ANGELES AP,4,11,9,9,18
4,23174,LOS ANGELES AP,5,10,10,11,21
5,23174,LOS ANGELES AP,6,10,9,11,20
6,23174,LOS ANGELES AP,7,13,5,13,18
7,23174,LOS ANGELES AP,8,14,5,12,17
8,23174,LOS ANGELES AP,9,13,6,10,16
9,23174,LOS ANGELES AP,10,13,8,10,18


---

In [28]:
sun_src = pd.read_fwf(
    "https://www.ncei.noaa.gov/sites/g/files/anmtlf171/files/2024-11/pctpos23.txt"
).dropna()

In [29]:
sun_src.columns = [
    "station",
    "period",
    "JAN",
    "FEB",
    "MAR",
    "APR",
    "MAY",
    "JUN",
    "JUL",
    "AUG",
    "SEP",
    "OCT",
    "NOV",
    "DEC",
    "ANN",
]

In [30]:
sun_src[
    [
        "JAN",
        "FEB",
        "MAR",
        "APR",
        "MAY",
        "JUN",
        "JUL",
        "AUG",
        "SEP",
        "OCT",
        "NOV",
        "DEC",
        "ANN",
    ]
] = sun_src[
    [
        "JAN",
        "FEB",
        "MAR",
        "APR",
        "MAY",
        "JUN",
        "JUL",
        "AUG",
        "SEP",
        "OCT",
        "NOV",
        "DEC",
        "ANN",
    ]
].astype(
    int
)

In [36]:
sun_src["station_code"] = sun_src["station"].str[:5].astype(str)

In [37]:
sun_src[["station_location", "station_state"]] = (
    sun_src["station"].str[5:].str.split(",", expand=True)
)

In [38]:
sun_src.columns = sun_src.columns.str.lower()

In [39]:
sun_df = sun_src[
    [
        "station_code",
        "station_location",
        "station_state",
        "jan",
        "feb",
        "mar",
        "apr",
        "may",
        "jun",
        "jul",
        "aug",
        "sep",
        "oct",
        "nov",
        "dec",
        "ann",
    ]
].dropna()

In [40]:
sun_la = sun_df.query('station_location.str.contains("LOS ANGELES")')

In [41]:
mean_sun_la = int(sun_la["ann"].iloc[0]) / 100

In [42]:
sun_la_melt = sun_la.melt(
    id_vars=["station_location"],
    value_vars=[
        "jan",
        "feb",
        "mar",
        "apr",
        "may",
        "jun",
        "jul",
        "aug",
        "sep",
        "oct",
        "nov",
        "dec",
    ],
    var_name="month",
    value_name="pct_sun",
)

In [43]:
sun_la_melt["pct_sun"] = sun_la_melt["pct_sun"] / 100

In [44]:
sun_la_melt["month_num"] = pd.to_datetime(
    sun_la_melt["month"], format="%b"
).dt.strftime("%-m")

In [45]:
base = alt.Chart(sun_la_melt).encode(
    x=alt.X("month_num:T", axis=alt.Axis(format="%b", grid=False), title=""),
    y=alt.Y(
        "pct_sun:Q",
        axis=alt.Axis(format="%", grid=False, offset=15),
        title="Mean possible sunshine",
    ),
    text=alt.Text("pct_sun:Q", format=".0%"),
)

bars = base.mark_bar(color="#f8c153", width=35)
text = base.mark_text(align="center", dy=-10, dx=0)

# Mean line
mean_line = (
    alt.Chart(pd.DataFrame({"y": [mean_sun_la]}))
    .mark_rule(color="#666", strokeDash=[4, 4])
    .encode(y="y:Q")
)

# Mean label
mean_label = (
    alt.Chart(pd.DataFrame({"y": [mean_sun_la], "text": ["Mean: 72%"]}))
    .mark_text(align="left", dx=-75, dy=-10, color="#666")
    .encode(y="y:Q", text="text:N")
)

chart = (
    (bars + text + mean_line + mean_label)
    .configure_view(strokeWidth=0)
    .properties(width=500, height=350, title="Percent sunshine in LA, by month")
)

chart

alt.LayerChart(...)

---

#### Observed

In [48]:
# Global Historical Climatology Network - Daily summaries — LAX (23174)
# https://www.ncei.noaa.gov/access/search/data-search/daily-summaries?bbox=35.294,-119.773,31.294,-115.773&pageNum=1&pageSize=10
daily_summaries_url = (
    "https://www.ncei.noaa.gov/data/daily-summaries/access/USW00023174.csv"
)

In [50]:
daily_df = pd.read_csv(daily_summaries_url, low_memory=False)

In [54]:
daily_df.tail(20)

,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,SNWD,SNWD_ATTRIBUTES,TMAX,TMAX_ATTRIBUTES,TMIN,TMIN_ATTRIBUTES,ACMH,ACMH_ATTRIBUTES,ACSH,ACSH_ATTRIBUTES,ADPT,ADPT_ATTRIBUTES,ASLP,ASLP_ATTRIBUTES,ASTP,ASTP_ATTRIBUTES,AWBT,AWBT_ATTRIBUTES,AWND,AWND_ATTRIBUTES,FMTM,FMTM_ATTRIBUTES,FRGT,FRGT_ATTRIBUTES,PGTM,PGTM_ATTRIBUTES,RHAV,RHAV_ATTRIBUTES,RHMN,RHMN_ATTRIBUTES,RHMX,RHMX_ATTRIBUTES,TAVG,TAVG_ATTRIBUTES,TSUN,TSUN_ATTRIBUTES,WDF1,WDF1_ATTRIBUTES,WDF2,WDF2_ATTRIBUTES,...,WSF1,WSF1_ATTRIBUTES,WSF2,WSF2_ATTRIBUTES,WSF5,WSF5_ATTRIBUTES,WSFG,WSFG_ATTRIBUTES,WSFI,WSFI_ATTRIBUTES,WSFM,WSFM_ATTRIBUTES,WT01,WT01_ATTRIBUTES,WT02,WT02_ATTRIBUTES,WT03,WT03_ATTRIBUTES,WT04,WT04_ATTRIBUTES,WT05,WT05_ATTRIBUTES,WT06,WT06_ATTRIBUTES,WT07,WT07_ATTRIBUTES,WT08,WT08_ATTRIBUTES,WT09,WT09_ATTRIBUTES,WT10,WT10_ATTRIBUTES,WT11,WT11_ATTRIBUTES,WT13,WT13_ATTRIBUTES,WT14,WT14_ATTRIBUTES,WT16,WT16_ATTRIBUTES,WT18,WT18_ATTRIBUTES,WT21,WT21_ATTRIBUTES,WV01,WV01_ATTRIBUTES,WV03,WV03_ATTRIBUTES,WV20,WV20_ATTRIBUTES
29791,USW00023174,2025-07-25,33.93816,-118.3866,29.7,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",0.0,",,W,2400",NaN,NaN,NaN,NaN,222.0,",,W",167.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.0,"H,,S",NaN,NaN,NaN,NaN,250.0,",,W",...,NaN,NaN,81.0,",,W",116.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29792,USW00023174,2025-07-26,33.93816,-118.3866,29.7,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",0.0,",,W,2400",NaN,NaN,NaN,NaN,228.0,",,W",167.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.0,"H,,S",NaN,NaN,NaN,NaN,260.0,",,W",...,NaN,NaN,89.0,",,W",121.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29793,USW00023174,2025-07-27,33.93816,-118.3866,29.7,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",0.0,",,W,2400",NaN,NaN,NaN,NaN,222.0,",,W",167.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,189.0,"H,,S",NaN,NaN,NaN,NaN,260.0,",,W",...,NaN,NaN,76.0,",,W",112.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29794,USW00023174,2025-07-28,33.93816,-118.3866,29.7,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",0.0,",,W,2400",NaN,NaN,NaN,NaN,228.0,",,W",172.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.0,"H,,S",NaN,NaN,NaN,NaN,240.0,",,W",...,NaN,NaN,81.0,",,W",112.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29795,USW00023174,2025-07-29,33.93816,-118.3866,29.7,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",0.0,",,W,2400",NaN,NaN,NaN,NaN,233.0,",,W",172.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.0,"H,,S",NaN,NaN,NaN,NaN,250.0,",,W",...,NaN,NaN,94.0,",,W",121.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29796,USW00023174,2025-07-30,33.93816,-118.3866,29.7,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",0.0,",,W,2400",NaN,NaN,NaN,NaN,233.0,",,W",172.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,197.0,"H,,S",NaN,NaN,NaN,NaN,260.0,",,W",...,NaN,NaN,81.0,",,W",107.0,",,W",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

---

## Exports

#### JSON

In [41]:
# df.to_json(
#     f"data/processed/NAME.json",
#     indent=4,
#     orient="records",
# )

#### CSV

In [42]:
# df.to_csv(
#     f"data/processed/NAME.csv", index=False
# )